<a href="https://colab.research.google.com/github/louistrue/learn-ifc/blob/main/BFH-25-PropertySets-full-Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧱 IFC PropertySets Dashboard

Interaktive Auswertung der PropertySets eines IFC-Modells direkt im Notebook. Aufbauend auf dem CSV-Export werden die Daten in einem Dash-Dashboard visualisiert.

**Outputs:**
- 📊 Interaktive Dash-Visualisierung direkt im Notebook
- 📄 CSV-Dateien wie im Export-Notebook (`property_sets_detailed.csv`, `property_sets_summary.csv`)


In [ ]:
%pip install ifcopenshell dash


In [ ]:
import ifcopenshell
import ifcopenshell.util.element as uel
import pandas as pd
from collections import defaultdict

from dash import Dash, dcc, html, Input, Output
import plotly.express as px


In [ ]:
# 📂 IFC-Modell laden
# Für Colab: Datei hochladen
# Für lokal: Pfad anpassen
try:
    from google.colab import files
    print("🔼 Google Colab erkannt - Bitte IFC-Datei hochladen...")
    uploaded = files.upload()
    ifc_filename = next(iter(uploaded))
except ModuleNotFoundError:
    ifc_filename = 'Modell.ifc'  # 👉 Pfad ggf. anpassen

print(f'📁 Verwende IFC-Datei: {ifc_filename}')
model = ifcopenshell.open(ifc_filename)


In [ ]:
# 📊 PropertySets extrahieren
detailed_data = []
summary_data = defaultdict(set)

print("🔍 Extrahiere PropertySets...")

for element in model.by_type('IfcElement'):
    element_id = element.GlobalId
    element_name = element.Name or ''
    element_type = element.is_a()

    psets = uel.get_psets(element)
    for pset_name, properties in psets.items():
        for prop_name, prop_value in properties.items():
            summary_data[pset_name].add(prop_name)
            detailed_data.append({
                'Element_ID': element_id,
                'Element_Name': element_name,
                'Element_Class': element_type,
                'PropertySet': pset_name,
                'Property': prop_name,
                'Value': prop_value
            })

print(f'✅ {len(detailed_data):,} Property-Einträge gesammelt.')


In [ ]:
# 💾 CSV Export: Detaillierte Daten
if detailed_data:
    df_detailed = pd.DataFrame(detailed_data)
    output_file = 'property_sets_detailed.csv'
    df_detailed.to_csv(output_file, index=False, encoding='utf-8')
    print(f'💾 Gespeichert: {output_file}')
else:
    raise ValueError('Keine Property-Daten gefunden. Bitte IFC-Modell prüfen.')


In [ ]:
# 💾 CSV Export: Zusammenfassung
if summary_data:
    summary_rows = [
        {'PropertySet': pset, 'Property': prop}
        for pset, props in summary_data.items()
        for prop in sorted(props)
    ]
    df_summary = pd.DataFrame(summary_rows)
    summary_file = 'property_sets_summary.csv'
    df_summary.to_csv(summary_file, index=False, encoding='utf-8')
    print(f'💾 Gespeichert: {summary_file}')
else:
    raise ValueError('Keine PropertySets gefunden. Bitte IFC-Modell prüfen.')


In [ ]:
# 📊 Statistiken
print("📊 Top 10 PropertySets:")
pset_counts = df_summary.groupby('PropertySet').size().sort_values(ascending=False)
for pset, count in pset_counts.head(10).items():
    print(f'   {count:>5}× {pset}')

print('
📊 Top 10 Properties:')
prop_counts = df_summary.groupby('Property').size().sort_values(ascending=False)
for prop, count in prop_counts.head(10).items():
    print(f'   {count:>5}× {prop}')


In [ ]:
# 🧮 Dashboard-Daten vorbereiten
if 'df_detailed' not in globals() or df_detailed.empty:
    raise ValueError('Keine detaillierten Daten verfügbar. Bitte vorherige Zellen ausführen.')

df_dash = df_detailed.copy()
df_dash['PropertySet'] = df_dash['PropertySet'].fillna('—')
df_dash['Property'] = df_dash['Property'].fillna('—')

agg = (
    df_dash.groupby(['PropertySet', 'Property'], dropna=False)
          .size()
          .reset_index(name='Count')
          .sort_values(['PropertySet', 'Count'], ascending=[True, False])
)

print(f'✅ Dashboard mit {len(agg):,} Einträgen vorbereitet.')


In [ ]:
# 📈 Dash-Dashboard starten
app = Dash(__name__)

app.layout = html.Div([
    html.H2('IFC PropertySet Dashboard'),
    html.Div([
        html.Label('PropertySet auswählen:'),
        dcc.Dropdown(
            id='pset-dropdown',
            options=[{'label': p, 'value': p} for p in agg['PropertySet'].unique()],
            value=agg['PropertySet'].iloc[0]
        ),
    ], style={'width': '40%', 'margin-bottom': '20px'}),
    dcc.Graph(id='bar-chart'),
    html.H3('Detailansicht'),
    html.Div(id='detail-table')
], style={'maxWidth': '1100px', 'margin': '0 auto', 'font-family': 'sans-serif'})

@app.callback(
    Output('bar-chart', 'figure'),
    Input('pset-dropdown', 'value')
)
def update_chart(selected_pset):
    sub = agg[agg['PropertySet'] == selected_pset]
    fig = px.bar(
        sub,
        x='Property',
        y='Count',
        title=f'Anzahl Properties in {selected_pset}',
        labels={'Property': 'Property', 'Count': 'Anzahl'},
    )
    fig.update_layout(
        margin=dict(l=20, r=20, t=60, b=60),
        xaxis_tickangle=-40
    )
    return fig

@app.callback(
    Output('detail-table', 'children'),
    Input('pset-dropdown', 'value')
)
def update_table(selected_pset):
    sub = df_dash[df_dash['PropertySet'] == selected_pset]
    top = sub.head(25)
    header = html.Tr([html.Th(col) for col in top.columns])
    rows = [
        html.Tr([html.Td(str(top.iloc[i][col])) for col in top.columns])
        for i in range(len(top))
    ]
    return html.Table([header] + rows, style={'width': '100%', 'border-collapse': 'collapse'})

app.run(jupyter_mode='inline', jupyter_height=600, jupyter_width='100%')
